# Does Yang's **Section 3** dump reproduce LLMGPR's Foursquare statistics?

**Why this notebook exists.** The previous notebook extracted from **Section 5**
(`dataset_WWW2019.zip`). That cannot work, and the reason is arithmetic, not tuning:

LLMGPR §4.1 removes POIs with fewer than 10 interactions, so its Table 1 requires
**80,962 POIs x 10 = 809,620 check-ins minimum**. The Section-5 extraction yields
**592,341** check-ins across the three cities *before any filter*. Filters only ever
remove check-ins, so no filter setting, group rule, or bounding box can close that gap.

LLMGPR cites its Foursquare data as `[4]` = Chen et al., IMWUT'20, whose dataset section
reads: *"33,278,683 check-in records of 266,909 users at 3,680,126 unique POIs between
April 2012 and September 2013 in the most checked 415 cities worldwide"*. That is verbatim
**Section 3** — a different user population (selected for activity, not for social
connectivity) with roughly 3x the local density.

**What this notebook does**

1. Pulls Section 3 and rebuilds LLMGPR Table 1 with a *real* iterated >=10 core.
2. Tries two city-assignment rules (bounding box, and nearest of Yang's 415 city centres)
   and reports both, since we do not know which one they used.
3. Section 3 ships **no friendship file**, so it probes whether Section-5 friendships can
   be attached to Section-3 check-ins — by fingerprinting users on `(venue_id, utc_time)`.

Runtime roughly 25-40 min on a Kaggle CPU notebook. Internet must be **on**.

## 0. Setup

In [ ]:
import os, sys, re, zipfile, subprocess, collections, random, gc
import pandas as pd, numpy as np

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-4000:])
    if p.stderr: print(p.stderr[-4000:], file=sys.stderr)
    if check and p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p

def fetch_drive(file_id, dest, expected_bytes=None, resourcekey=None):
    """Download a public Drive file and verify it.

    gdown's fuzzy URL parser normalises the link to `uc?id=...` and DROPS the
    `resourcekey` query param, which the legacy (0Bwrg...) ids require -> 403 HTML.
    drive.usercontent.google.com honours resourcekey + confirm=t, so use it directly.
    subprocess.run without capture streams curl's progress bar into the cell."""
    if os.path.exists(dest) and (expected_bytes is None
                                 or os.path.getsize(dest) == expected_bytes):
        print(f"already have {dest} ({os.path.getsize(dest)/1024**3:.2f} GB)")
    else:
        url = (f"https://drive.usercontent.google.com/download?id={file_id}"
               f"&export=download&confirm=t"
               + (f"&resourcekey={resourcekey}" if resourcekey else ""))
        print("$ curl", url, flush=True)
        rc = subprocess.run(f'curl -L --fail --retry 3 --retry-delay 5 -o "{dest}" "{url}"',
                            shell=True).returncode
        if rc: raise RuntimeError(f"curl failed (exit {rc})")

    size = os.path.getsize(dest)
    with open(dest, "rb") as f:
        magic = f.read(4)
    assert magic == b"PK\x03\x04", (
        f"{dest} is not a zip (starts {magic!r}) - Drive served an HTML page. "
        "Download it by hand and attach it as a Kaggle Dataset instead.")
    if expected_bytes and size != expected_bytes:
        print(f"WARNING: got {size:,} bytes, expected {expected_bytes:,}")
    with zipfile.ZipFile(dest) as z:          # validates the central directory
        names = [n for n in z.namelist() if not n.endswith("/") and "__MACOSX" not in n]
    print(f"OK {dest}: {size/1024**3:.2f} GB, {len(names)} entries")
    return dest

def save_table(df, stem):
    """parquet when an engine is available, csv otherwise."""
    try:
        df.to_parquet(f"{stem}.parquet", index=False); out = f"{stem}.parquet"
    except ImportError:
        df.to_csv(f"{stem}.csv", index=False); out = f"{stem}.csv"
    print("wrote", out)
    return out

def du():
    sh("df -h /kaggle/working | tail -1", check=False)

print("pandas", pd.__version__, "| numpy", np.__version__)

# ---- LLMGPR targets, used for every comparison below --------------------------
TARGET_3CITY = dict(users=7_507, groups=1_715, pois=80_962, cats=436,
                    checkins=1_214_631, group_checkins=12_594)          # CIKM'25
TARGET_NYC   = dict(users=6_078, groups=1_557, pois=63_445, cats=436,
                    checkins=923_856, group_checkins=10_899)            # arXiv v1
MIN_INTERACTIONS = 10        # LLMGPR 4.1: "less than 10 interactions are removed"

## 1. Download Section 3 - *Global-scale Check-in Dataset*

33,278,683 check-ins / 266,909 users / 3,680,126 venues / 415 cities, Apr 2012 - Sep 2013.
Ships as `dataset_TIST2015.zip`, 739.8 MB, containing `Checkins` / `POIs` / `Cities` / `readme`.

**Why not gdown.** This is a legacy Drive id that only serves with its `resourcekey`
attached, and gdown's fuzzy URL parser normalises the link to `uc?id=...`, dropping the
resourcekey - Drive then answers with a 403 HTML page and gdown raises
`FileURLRetrievalError`. `drive.usercontent.google.com` honours `resourcekey` plus
`confirm=t`, so `fetch_drive()` calls it with curl and verifies the result
(byte count, `PK\x03\x04` magic, readable zip central directory).

If it still fails, download the zip by hand from
<https://sites.google.com/site/yangdingqi/home/foursquare-dataset> (item 3), attach it as a
Kaggle Dataset, and point `TIST_ZIP` at the attached path - the rest of the notebook is
indifferent to how the file arrived.

In [ ]:
TIST_ZIP = f"{WORK}/tist2015.zip"

# Section 3 - "Global-scale Check-in Dataset" (dataset_TIST2015.zip, 739.8 MB).
# Legacy Drive id: it only serves with its resourcekey attached.
fetch_drive("0BwrgZ-IdrTotZ0U0ZER2ejI3VVk", TIST_ZIP,
            expected_bytes=775_746_915,
            resourcekey="0-rlHp_JcRyFAxN7v5OAGldw")


In [ ]:
with zipfile.ZipFile(TIST_ZIP) as z:
    for i in z.infolist():
        if not i.filename.endswith("/"):
            print(f"{i.filename:<60} {i.file_size/1024**2:8.1f} MB")
    z.extractall(WORK)

def find(pat, root=WORK):
    hits = []
    for dp, _, fns in os.walk(root):
        if "__MACOSX" in dp: continue
        for fn in fns:
            if re.search(pat, fn, re.I) and not fn.startswith("._"):
                hits.append(os.path.join(dp, fn))
    return sorted(hits)

# exclude the Section-5 files, in case this cell is re-run after section 5 is fetched
_no5 = lambda ps: [p for p in ps if "WWW" not in p and "raw_" not in os.path.basename(p)]
TIST_CHECKINS = _no5(find(r"checkin.*\.txt$"))[0]
TIST_POIS     = _no5(find(r"poi.*\.txt$"))[0]
_cities       = find(r"cit(y|ies).*\.txt$")
TIST_CITIES   = _cities[0] if _cities else None
print("\ncheck-ins:", TIST_CHECKINS)
print("POIs     :", TIST_POIS)
print("cities   :", TIST_CITIES)

for p in find(r"readme"):
    print("\n" + "="*80 + f"\n{p}\n" + "="*80)
    print(open(p, encoding="utf-8", errors="replace").read()[:4000])

os.remove(TIST_ZIP); du()   # reclaim 775 MB - /kaggle/working is capped at 20 GB

## 2. POIs and city assignment

Two rules, reported side by side, because we don't know which one LLMGPR used:

- **bbox** — the same three boxes the previous notebook used, so the numbers stay comparable.
- **centre** — nearest of Yang's own 415 city centres within a radius, swept over several
  radii. If one radius reproduces 63,445 NYC POIs, that is very likely their rule.

In [ ]:
CITY_BBOX = {
    "New York":    dict(lon_min=-74.3,  lon_max=-73.6,  lat_min=40.4, lat_max=41.0),
    "Chicago":     dict(lon_min=-88.0,  lon_max=-87.5,  lat_min=41.6, lat_max=42.1),
    "Los Angeles": dict(lon_min=-118.7, lon_max=-117.6, lat_min=33.6, lat_max=34.4),
}

pois = pd.read_csv(TIST_POIS, sep="\t", header=None,
                   names=["venue_id", "lat", "lon", "category", "country"],
                   dtype={"venue_id": str}, on_bad_lines="skip", low_memory=False)
pois["lat"] = pd.to_numeric(pois["lat"], errors="coerce")
pois["lon"] = pd.to_numeric(pois["lon"], errors="coerce")
pois = pois.dropna(subset=["lat", "lon"])
print(f"POIs: {len(pois):,} | categories: {pois['category'].nunique():,} "
      f"(IMWUT'20 says 429 for this dump; LLMGPR reports 436)")

VENUE2CAT = dict(zip(pois["venue_id"], pois["category"]))

bbox_venues = {}
for city, b in CITY_BBOX.items():
    m = (pois["lon"].between(b["lon_min"], b["lon_max"]) &
         pois["lat"].between(b["lat_min"], b["lat_max"]))
    bbox_venues[city] = set(pois.loc[m, "venue_id"])
    print(f"  bbox {city:<12}: {len(bbox_venues[city]):,} venues")
print(f"  bbox TOTAL       : {sum(map(len, bbox_venues.values())):,}")

In [ ]:
# Nearest-city-centre assignment. The Cities.txt schema is unverified until the readme
# above is read, so this is best-effort and simply skips if the file doesn't parse.
centre_venues = {}
try:
    cities = pd.read_csv(TIST_CITIES, sep="\t", header=None, on_bad_lines="skip")
    cities.columns = (["city", "lat", "lon", "country_code", "country", "type"]
                      if cities.shape[1] == 6 else
                      [f"c{i}" for i in range(cities.shape[1])])
    print(cities.head(3).to_string(), "\n")
    want = {"New York": r"^new york", "Chicago": r"^chicago", "Los Angeles": r"^los angeles"}
    centres = {}
    for city, pat in want.items():
        hit = cities[cities["city"].astype(str).str.match(pat, case=False, na=False)]
        if len(hit):
            centres[city] = (float(hit.iloc[0]["lat"]), float(hit.iloc[0]["lon"]))
            print(f"  centre {city:<12}: {centres[city]}")

    lat = pois["lat"].to_numpy(); lon = pois["lon"].to_numpy()
    for R in (15, 25, 40, 60):
        row = []
        for city, (clat, clon) in centres.items():
            d = np.hypot((lat - clat) * 111.32,
                         (lon - clon) * 111.32 * np.cos(np.radians(clat)))
            sel = set(pois.loc[d <= R, "venue_id"])
            if R == 40: centre_venues[city] = sel
            row.append(f"{city}={len(sel):,}")
        print(f"  R={R:>3} km -> " + "  ".join(row))
except Exception as e:
    print("centre assignment skipped:", type(e).__name__, e)

## 3. Stream the 33.3M check-ins, keep the three cities

In [ ]:
CK_COLS = ["user_id", "venue_id", "utc_time", "tz_offset"]

def read_checkins(path, **kw):
    return pd.read_csv(path, sep="\t", header=None, names=CK_COLS,
                       dtype={"user_id": str, "venue_id": str, "utc_time": str},
                       usecols=[0, 1, 2, 3], on_bad_lines="skip", **kw)

keep = pd.Index(sorted(set().union(*bbox_venues.values(), *centre_venues.values())))
print(f"target venue set: {len(keep):,}")

parts, seen = [], 0
for ch in read_checkins(TIST_CHECKINS, chunksize=2_000_000):
    seen += len(ch)
    parts.append(ch[ch["venue_id"].isin(keep)])
    print(f"\rscanned {seen:,}", end="", flush=True)
ck = pd.concat(parts, ignore_index=True)
del parts; gc.collect()

print(f"\n\nin-scope check-ins (unfiltered): {len(ck):,}")
print(f"distinct users {ck['user_id'].nunique():,} | distinct venues {ck['venue_id'].nunique():,}")
save_table(ck, f"{WORK}/tist_3city_raw")

## 4. LLMGPR Table 1, rebuilt

In [ ]:
def k_core(df, k=MIN_INTERACTIONS, verbose=False):
    # Iterated bipartite k-core: keep users AND POIs with >= k check-ins. Iterated
    # because dropping a user can push a POI under the threshold, and vice versa.
    n0 = len(df)
    while True:
        before = len(df)
        vc = df["user_id"].value_counts();  df = df[df["user_id"].isin(vc[vc >= k].index)]
        vc = df["venue_id"].value_counts(); df = df[df["venue_id"].isin(vc[vc >= k].index)]
        if len(df) == before or df.empty: break
    if verbose: print(f"    {k}-core: {n0:,} -> {len(df):,}")
    return df

def table1(assign, label):
    print(f"\n### city assignment = {label}   (>= {MIN_INTERACTIONS} interactions)")
    print(f"{'city':<15}{'users':>9}{'POIs':>9}{'cats':>7}{'check-ins':>13}{'ck/user':>9}")
    per = {}
    for city, vset in assign.items():
        d = k_core(ck[ck["venue_id"].isin(vset)].copy())
        per[city] = d
        cats = pd.Series([VENUE2CAT.get(v) for v in d["venue_id"].unique()]).nunique()
        u = max(d["user_id"].nunique(), 1)
        print(f"{city:<15}{d['user_id'].nunique():>9,}{d['venue_id'].nunique():>9,}"
              f"{cats:>7,}{len(d):>13,}{len(d)/u:>9.1f}")
    allc = pd.concat(per.values(), ignore_index=True)
    cats = pd.Series([VENUE2CAT.get(v) for v in allc["venue_id"].unique()]).nunique()
    u = max(allc["user_id"].nunique(), 1)
    print(f"{'ALL 3':<15}{allc['user_id'].nunique():>9,}{allc['venue_id'].nunique():>9,}"
          f"{cats:>7,}{len(allc):>13,}{len(allc)/u:>9.1f}")
    for name, t in (("TARGET 3city", TARGET_3CITY), ("TARGET NYConly", TARGET_NYC)):
        print(f"{name:<15}{t['users']:>9,}{t['pois']:>9,}{t['cats']:>7,}"
              f"{t['checkins']:>13,}{t['checkins']/t['users']:>9.1f}")
    return per, allc

per_bbox, all_bbox = table1(bbox_venues, "bounding box")
if centre_venues:
    per_ctr, all_ctr = table1(centre_venues, "nearest city centre, R=40 km")

# Keep the bbox version for downstream work.
save_table(all_bbox, f"{WORK}/tist_3city_10core")

In [ ]:
# Plain-English readout of whether Section 3 can carry LLMGPR's table at all.
n_ck = len(ck)
need = TARGET_3CITY["pois"] * MIN_INTERACTIONS
print(f"unfiltered in-scope check-ins : {n_ck:,}")
print(f"floor implied by their table  : {need:,}  ({TARGET_3CITY['pois']:,} POIs x {MIN_INTERACTIONS})")
if n_ck >= need:
    print(f"=> FEASIBLE. Section 3 has {n_ck/need:.2f}x the minimum. "
          "Compare the POI columns above and pick the assignment rule that matches.")
else:
    print(f"=> STILL SHORT by {need/n_ck:.2f}x. Section 3 is not their source either; "
          "stop chasing their table and report our own statistics honestly.")
print(f"\nfor reference, Section 5 gave 592,341 check-ins / 102,541 ever-visited POIs")

## 5. Can Section-5 friendships be attached to Section-3 check-ins?

Section 3 has no friendship file, and CubeRec-style group construction needs social edges.
The two dumps are anonymised **independently**, so the question is whether a user can be
matched across them. A `(venue_id, utc_time)` pair is near-unique across 3.7M venues at
second resolution, so it works as a fingerprint: sample Section-5 users who carry a
friendship edge, collect their check-in fingerprints, then stream Section 3 and see which
Section-3 user those fingerprints land on.

Three outcomes:

- **same ids** → the dumps share one id space; join on `user_id` and move on.
- **different ids, purity ~1.0** → build the id map from the votes and carry friendships over.
- **few matches / low purity** → genuinely disjoint; we need a different social source.

In [ ]:
WWW_ZIP = f"{WORK}/dataset_WWW2019.zip"
if not find(r"WWW_Checkins.*\.txt$"):
    # Section 5 - "...with User Social Networks" (2.50 GB). No resourcekey on this one.
    fetch_drive("1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8", WWW_ZIP,
                expected_bytes=2_684_000_558)

if os.path.exists(WWW_ZIP):
    with zipfile.ZipFile(WWW_ZIP) as z:
        # only the anonymised check-ins + friendships; skip raw_POIs (672 MB)
        # and raw_Checkins (5.8 GB), neither of which this probe needs
        wanted = [n for n in z.namelist()
                  if "__MACOSX" not in n and not n.endswith("/")
                  and (("friend" in n.lower())
                       or ("checkin" in n.lower() and "raw" not in n.lower()))]
        print("extracting:", wanted)
        for n in wanted: z.extract(n, WORK)
    os.remove(WWW_ZIP)

S5_CHECKINS = find(r"WWW_Checkins.*\.txt$")[0]
FRIEND_OLD  = find(r"friendship_old.*\.txt$")[0]
print("S5 check-ins:", S5_CHECKINS)
print("friend old  :", FRIEND_OLD)
du()

In [ ]:
SAMPLE_N = 2000
random.seed(0)

e = pd.read_csv(FRIEND_OLD, sep="\t", header=None, names=["u", "v"], dtype=str)
s5_users = set(e["u"]) | set(e["v"])
print(f"S5 users with >=1 friendship_old edge: {len(s5_users):,}")

sample = set(random.sample(sorted(s5_users), min(SAMPLE_N, len(s5_users))))
fps = []
for ch in read_checkins(S5_CHECKINS, chunksize=2_000_000):
    fps.append(ch.loc[ch["user_id"].isin(sample), ["user_id", "venue_id", "utc_time"]])
fp = (pd.concat(fps, ignore_index=True)
        .rename(columns={"user_id": "s5_user"})
        .drop_duplicates(subset=["venue_id", "utc_time"]))
del fps; gc.collect()
print(f"fingerprints from {len(sample):,} sampled users: {len(fp):,} unique (venue, time) pairs")

In [ ]:
votes = collections.defaultdict(collections.Counter)
hits, seen = 0, 0
for ch in read_checkins(TIST_CHECKINS, chunksize=2_000_000):
    seen += len(ch)
    m = ch.merge(fp, on=["venue_id", "utc_time"], how="inner")
    hits += len(m)
    for s5, s3 in zip(m["s5_user"], m["user_id"]):
        votes[s5][s3] += 1
    print(f"\rscanned {seen:,} | fingerprint hits {hits:,}", end="", flush=True)

matched = [u for u in votes if votes[u]]
identical = sum(1 for u in matched if votes[u].most_common(1)[0][0] == u)
purity = sorted(votes[u].most_common(1)[0][1] / sum(votes[u].values()) for u in matched)
med = purity[len(purity)//2] if purity else 0.0

print(f"\n\nS5 users found in S3 : {len(matched):,} / {len(sample):,} ({len(matched)/len(sample):.1%})")
print(f"  ...with the SAME id: {identical:,}")
print(f"  median vote purity : {med:.2f}")
print(f"  purity >= 0.9      : {sum(1 for x in purity if x >= 0.9):,}")
print("\nexamples (s5_id -> top s3_id, votes):")
for u in matched[:10]:
    top, n = votes[u].most_common(1)[0]
    print(f"  {u} -> {top}  ({n}/{sum(votes[u].values())})")

if matched and identical >= 0.9 * len(matched):
    print("\nVERDICT: shared id space - join Section-5 friendships on user_id directly.")
elif matched and med >= 0.9:
    print("\nVERDICT: distinct ids but a clean 1:1 map - build s5->s3 from these votes, "
          "then carry friendship_old across. (friendship_old ONLY: friendship_new "
          "postdates the check-in window and leaks.)")
else:
    print("\nVERDICT: no usable correspondence - Section-3 check-ins cannot inherit "
          "Section-5 friendships. Report this before choosing a social source.")

pd.DataFrame([(u, votes[u].most_common(1)[0][0], votes[u].most_common(1)[0][1],
               sum(votes[u].values())) for u in matched],
             columns=["s5_user", "s3_user", "top_votes", "total_votes"]
            ).to_csv(f"{WORK}/s5_to_s3_idmap_sample.csv", index=False)
print(f"\nwrote {WORK}/s5_to_s3_idmap_sample.csv")

## What to send back

1. Both **Table 1** blocks from section 4 (bbox and centre), plus the radius sweep.
2. The **FEASIBLE / STILL SHORT** line.
3. The **VERDICT** block from section 5.

Group construction is deliberately *not* in this notebook. The previous run produced 20,676
co-presence events over 17,701 distinct member-sets — **1.17 check-ins per group against
LLMGPR's 7.34** — so leave-one-out (which needs >=3 per group) is impossible on it, and at
most 1,487 of those sets could ever reach 3. That stage needs rewriting regardless of which
dataset wins: persistent member-sets with pooled sequences, a rolling time window instead of
12-hour `floor` buckets, cliques instead of connected components, and `friendship_old` only.
Worth doing once, on whichever data these results select.